# 🏨 Iceberg Property Combined — Explorer Notebook

**Uses**: `pyspark.sql` → Iceberg catalog → `rental_property` + `property_reviews`  
**View**: `property_combined` (inner join on `rental_property.id == property_reviews.gen_id`)  
**Catalog**: Hadoop FileSystem (`./warehouse`)


## 0 · Setup — SparkSession with Iceberg

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Resolve repo root whether notebook lives at repo root or notebooks/
NOTEBOOK_DIR = os.path.abspath('')
if os.path.exists(os.path.join(NOTEBOOK_DIR, 'config.py')):
    PROJECT_DIR = NOTEBOOK_DIR
else:
    PROJECT_DIR = os.path.dirname(NOTEBOOK_DIR)
WAREHOUSE = os.path.join(PROJECT_DIR, 'warehouse')

CATALOG = 'local'
RENTAL_TABLE  = f'{CATALOG}.property_db.rental_property'
REVIEWS_TABLE = f'{CATALOG}.property_db.property_reviews'
COMBINED_VIEW = 'property_combined'

spark = (
    SparkSession.builder
    .appName('IcebergViewer')
    .master('local[*]')
    .config(
        'spark.sql.extensions',
        'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions'
    )
    .config(f'spark.sql.catalog.{CATALOG}',
            'org.apache.iceberg.spark.SparkCatalog')
    .config(f'spark.sql.catalog.{CATALOG}.type', 'hadoop')
    .config(f'spark.sql.catalog.{CATALOG}.warehouse', WAREHOUSE)
    .config(
        'spark.jars.packages',
        'org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2'
    )
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('✅ SparkSession ready — Iceberg catalog:', CATALOG)
print('   Warehouse:', WAREHOUSE)
print('   Tables:', RENTAL_TABLE, 'and', REVIEWS_TABLE)


26/03/16 13:19:31 WARN Utils: Your hostname, w3e39 resolves to a loopback address: 127.0.1.1; using 192.168.0.227 instead (on interface enp2s0)
26/03/16 13:19:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/w3e39/Downloads/property_pipeline/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/w3e39/.ivy2/cache
The jars for the packages stored in: /home/w3e39/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.4_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c7ffa7ea-205f-46cb-8478-000d8eabe19b;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.4_2.12;1.4.2 in central
:: resolution report :: resolve 130ms :: artifacts dl 15ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.4_2.12;1.4.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spa

✅ SparkSession ready — Iceberg catalog: local
   Warehouse: /home/w3e39/Downloads/warehouse


## 1 · Schema & Row Count

In [2]:
# Load base Iceberg tables
df_rental  = spark.table(RENTAL_TABLE)
df_reviews = spark.table(REVIEWS_TABLE)

# Build combined view (rental + reviews)
df_combined = (
    df_rental.alias('r')
    .join(df_reviews.alias('v'), F.col('r.id') == F.col('v.gen_id'), how='inner')
    .select(
        F.col('r.id').alias('id'),
        F.col('r.feed_provider_id'),
        F.col('r.property_name'),
        F.col('r.property_slug'),
        F.col('r.country_code'),
        F.col('r.currency'),
        F.col('r.usd_price'),
        F.col('r.star_rating'),
        F.col('r.review_score'),
        F.col('r.commission'),
        F.col('r.meal_plan'),
        F.col('r.published'),
        F.col('r.data_quality_flag'),
        F.col('v.property_id'),
        F.col('v.review_id'),
        F.col('v.review_date'),
        F.col('v.review_year'),
        F.col('v.review_individual_score'),
        F.col('v.review_language'),
        F.col('v.review_summary'),
        F.col('v.review_positive'),
        F.col('v.review_negative'),
        F.col('v.reviewer_name'),
        F.col('v.reviewer_country'),
        F.col('v.reviewer_travel_purpose'),
        F.col('v.reviewer_type'),
    )
)

df_combined.createOrReplaceTempView(COMBINED_VIEW)

print(f'Combined rows : {df_combined.count()}')
print(f'Columns       : {len(df_combined.columns)}')
print('
Schema:')
df_combined.printSchema()


AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `local`.`property_db`.`property_reviews` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.;
'UnresolvedRelation [local, property_db, property_reviews], [], false


26/03/16 13:19:51 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


## 2 · Sample Rows

In [ ]:
# Register as a temp view so we can use plain SQL
spark.sql(f"""
    SELECT id, property_name, country_code,
           star_rating, review_score, review_year,
           reviewer_name, review_individual_score, data_quality_flag
    FROM   {COMBINED_VIEW}
    LIMIT  10
""").show(truncate=False)


## 3 · Reviews per Country


In [ ]:
spark.sql(f"""
    SELECT  country_code,
            COUNT(DISTINCT id)          AS properties,
            COUNT(review_id)            AS total_reviews,
            ROUND(AVG(review_individual_score), 2) AS avg_review_score,
            ROUND(AVG(star_rating), 2)  AS avg_stars
    FROM    {COMBINED_VIEW}
    GROUP BY country_code
    ORDER BY total_reviews DESC
""").show(40, truncate=False)


## 4 · Review Volume by Year


In [ ]:
spark.sql(f"""
    SELECT  review_year,
            COUNT(*)                         AS total_review_rows,
            ROUND(AVG(review_individual_score), 2) AS avg_score,
            COUNT(DISTINCT country_code)     AS countries_active
    FROM    {COMBINED_VIEW}
    WHERE   review_year IS NOT NULL
    GROUP BY review_year
    ORDER BY review_year
""").show()


## 5 · Top 10 Properties by Average Review Score

In [ ]:
spark.sql(f"""
    SELECT  id,
            FIRST(property_name)                       AS name,
            FIRST(country_code)                        AS country,
            FIRST(star_rating)                         AS stars,
            ROUND(AVG(review_individual_score), 2)     AS avg_score,
            COUNT(review_id)                           AS num_reviews
    FROM    {COMBINED_VIEW}
    WHERE   review_id IS NOT NULL
    GROUP BY id
    HAVING  num_reviews >= 2
    ORDER BY avg_score DESC
    LIMIT   10
""").show(truncate=False)


## 6 · Data Quality Distribution

In [ ]:
spark.sql(f"""
    SELECT  data_quality_flag,
            COUNT(DISTINCT id) AS properties,
            COUNT(*)            AS rows
    FROM    {COMBINED_VIEW}
    GROUP BY data_quality_flag
""").show()


## 7 · Pricing & Commercials


In [ ]:
# Pricing and commercial signals at property level (deduplicated)
spark.sql(f"""
    WITH property_base AS (
        SELECT id, property_name, country_code, currency, usd_price, commission, meal_plan, published
        FROM {COMBINED_VIEW}
        GROUP BY id, property_name, country_code, currency, usd_price, commission, meal_plan, published
    )
    SELECT country_code, currency,
           COUNT(*) AS properties,
           ROUND(AVG(usd_price), 2) AS avg_usd_price,
           ROUND(MIN(usd_price), 2) AS min_usd_price,
           ROUND(MAX(usd_price), 2) AS max_usd_price
    FROM property_base
    GROUP BY country_code, currency
    ORDER BY properties DESC
""").show(40, truncate=False)

spark.sql(f"""
    WITH property_base AS (
        SELECT id, usd_price, commission, meal_plan, published
        FROM {COMBINED_VIEW}
        GROUP BY id, usd_price, commission, meal_plan, published
    )
    SELECT meal_plan,
           COUNT(*) AS properties,
           ROUND(AVG(usd_price), 2) AS avg_usd_price,
           ROUND(AVG(commission), 4) AS avg_commission,
           SUM(CASE WHEN commission IS NULL THEN 1 ELSE 0 END) AS missing_commission
    FROM property_base
    GROUP BY meal_plan
    ORDER BY properties DESC
""").show(truncate=False)

spark.sql(f"""
    WITH property_base AS (
        SELECT id, property_name, usd_price, commission, published
        FROM {COMBINED_VIEW}
        GROUP BY id, property_name, usd_price, commission, published
    )
    SELECT id, property_name, usd_price, commission, published
    FROM property_base
    ORDER BY usd_price DESC
    LIMIT 10
""").show(truncate=False)


## 8 · Reviewer Travel Purpose Breakdown


In [ ]:
spark.sql(f"""
    SELECT  reviewer_travel_purpose,
            COUNT(*)                              AS reviews,
            ROUND(AVG(review_individual_score),2) AS avg_score
    FROM    {COMBINED_VIEW}
    WHERE   reviewer_travel_purpose IS NOT NULL
    GROUP BY reviewer_travel_purpose
    ORDER BY reviews DESC
""").show()


## 9 · Join Coverage & Review Density


In [ ]:
spark.sql(f"""
    SELECT
        COUNT(DISTINCT id) AS properties_total,
        COUNT(DISTINCT CASE WHEN review_id IS NOT NULL THEN id END) AS properties_with_reviews,
        COUNT(DISTINCT CASE WHEN review_id IS NULL THEN id END) AS properties_without_reviews,
        COUNT(review_id) AS total_reviews
    FROM {COMBINED_VIEW}
""").show(truncate=False)

spark.sql(f"""
    SELECT  id,
            COUNT(review_id) AS reviews_per_property
    FROM    {COMBINED_VIEW}
    GROUP BY id
    ORDER BY reviews_per_property DESC
    LIMIT   10
""").show(truncate=False)


## 10 · Filtered Reviews (GB, 2024)


In [ ]:
# Filtered scan over country + review_year
spark.sql(f"""
    SELECT  property_name, reviewer_name,
            review_date, review_individual_score, review_summary
    FROM    {COMBINED_VIEW}
    WHERE   country_code = 'GB'
      AND   review_year  = 2024
    ORDER BY review_individual_score DESC
""").show(truncate=False)


## 11 · Reviewer Type vs Star Rating Correlation


In [ ]:
spark.sql(f"""
    SELECT  reviewer_type,
            ROUND(AVG(star_rating), 2)             AS avg_stars,
            ROUND(AVG(review_individual_score), 2) AS avg_review_score,
            COUNT(*)                               AS total_reviews
    FROM    {COMBINED_VIEW}
    WHERE   reviewer_type IS NOT NULL
    GROUP BY reviewer_type
    ORDER BY avg_review_score DESC
""").show(truncate=False)


## 12 · Matplotlib Charts (optional)


In [ ]:
try:
    import matplotlib.pyplot as plt
    import pandas as pd

    # ── Chart 1: Reviews per country ─────────────────────────────────────────
    country_pd = spark.sql(f"""
        SELECT country_code, COUNT(review_id) AS reviews
        FROM   {COMBINED_VIEW}
        WHERE  review_id IS NOT NULL
        GROUP BY country_code
        ORDER BY reviews DESC
        LIMIT 15
    """).toPandas()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].bar(country_pd['country_code'], country_pd['reviews'], color='steelblue')
    axes[0].set_title('Reviews per Country (top 15)')
    axes[0].set_xlabel('Country Code')
    axes[0].set_ylabel('Number of Reviews')
    axes[0].tick_params(axis='x', rotation=45)

    # ── Chart 2: Avg score per year ───────────────────────────────────────────
    year_pd = spark.sql(f"""
        SELECT  review_year,
                ROUND(AVG(review_individual_score), 2) AS avg_score
        FROM    {COMBINED_VIEW}
        WHERE   review_year IS NOT NULL
        GROUP BY review_year
        ORDER BY review_year
    """).toPandas()

    axes[1].plot(year_pd['review_year'], year_pd['avg_score'],
                 marker='o', linewidth=2, color='darkorange')
    axes[1].set_title('Average Review Score by Year')
    axes[1].set_xlabel('Year')
    axes[1].set_ylabel('Avg Score')
    axes[1].set_ylim(0, 10)
    axes[1].grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.savefig(os.path.join(PROJECT_DIR, 'notebooks', 'iceberg_charts.png'),
                dpi=120, bbox_inches='tight')
    plt.show()
    print('Charts saved → notebooks/iceberg_charts.png')

except ImportError:
    print('matplotlib not installed — skipping charts (pip install matplotlib pandas)')


## 13 · Clean Up


In [ ]:
spark.stop()
print('SparkSession stopped.')